Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_DIR = "../data"

"NaT" text is the dataset-wide missing placeholder, so train and test are loaded identically with `na_values=["NaT"]`.

In [ ]:
train_raw = pd.read_excel(f"{DATA_DIR}/Assignment_Train.xlsx", na_values=["NaT"])
test_raw  = pd.read_excel(f"{DATA_DIR}/Assignment_Test.xlsx",  na_values=["NaT"])

assert train_raw.shape == (10_000, 94)
assert test_raw.shape == (5_000, 93)
assert list(test_raw.columns) == [c for c in train_raw.columns if c != "Target"]
assert train_raw["UID"].is_unique and test_raw["UID"].is_unique
print("train:", train_raw.shape, "| test:", test_raw.shape)

Drop the 8 rows without a label; never impute the target. Target = 1 is assumed to mean delinquent.

In [ ]:
has_label = train_raw["Target"].notna()
train = train_raw.loc[has_label].copy()
train["Target"] = train["Target"].astype("int8")

assert len(train) == 9_992
assert set(train["Target"].unique()) == {0, 1}
print("Rows dropped:", int((~has_label).sum()), "| positives:", int(train["Target"].sum()),
      "| positive rate:", round(train["Target"].mean(), 4))

Stratified 70:30, fixed seed, done before any target-based work. The holdout is used **once**, for the final estimate.

In [ ]:
dev, holdout = train_test_split(
    train, test_size=0.30, stratify=train["Target"], random_state=RANDOM_STATE
)

assert len(dev) == 6_994 and len(holdout) == 2_998
assert set(dev["UID"]).isdisjoint(holdout["UID"])
print("dev:", dev.shape, round(dev["Target"].mean(), 4), "| holdout:", holdout.shape, round(holdout["Target"].mean(), 4))

In [ ]:
SENTINEL_COLS = [
    "Email Attributes.digitalAge",
    "Email Name Attributes.digitalage",
    "Phone Name Attributes.phoneNameDigitalAge",
    "Email Name Attributes.nameMatchScore",
]
NOT_FOUND_FLAGS = {                                   # flag name -> column it is built from
    "email_name_not_found": "Email Name Attributes.nameMatchScore",
    "email_not_found":      "Email Name Attributes.digitalage",
    "phone_name_not_found": "Phone Name Attributes.phoneNameDigitalAge",
}

def handle_sentinels(data, mode="flag"):
    """mode='raw': leave -1 as-is (option a). mode='flag': add not-found flags, then -1 -> NaN (option c)."""
    out = data.copy()
    if mode == "raw":
        return out
    if mode != "flag":
        raise ValueError(f"unknown mode: {mode}")

    # 1. flags first, while the -1 values still exist
    for flag, source in NOT_FOUND_FLAGS.items():
        out[flag] = out[source].eq(-1).astype("int8")

    # 2. then replace -1 with NaN (the real values stay untouched)
    out[SENTINEL_COLS] = out[SENTINEL_COLS].replace(-1, np.nan)
    return out

dev_s = handle_sentinels(dev)
print(dev_s[list(NOT_FOUND_FLAGS)].mean().round(3))                 # expect ~0.749, 0.229, 0.589
print("-1 left in sentinel cols:", int(dev_s[SENTINEL_COLS].eq(-1).sum().sum()))   # expect 0

After replacing −1 with NaN, there are no −1 values left, so .eq(-1) would find nothing and every flag would be 0. The flags must be built while the −1s still exist.

In [ ]:
def add_platform_share(data):
    """Share of known platforms where the person is registered, separately for email and phone."""
    out = data.copy()
    for source in ["Email", "Phone"]:
        cols = [c for c in data.columns
                if c.startswith(f"{source} Social Advance.")
                and not c.endswith((".paytm", ".skype"))]
        n_known   = data[cols].notna().sum(axis=1)   # platforms with a 0 or 1
        n_present = data[cols].sum(axis=1)           # number of 1s (NaN is skipped)
        out[f"{source.lower()}_platform_known"] = n_known
        out[f"{source.lower()}_platform_share"] = n_present / n_known.replace(0, np.nan)
    return out

dev_p = add_platform_share(dev_s)
print(dev_p[["email_platform_known", "email_platform_share"]].median().round(2))

In [ ]:
ORDINAL_MAPS = {
    "Email Attributes.footprint": {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "NO_RECORDS": np.nan},
    "Phone Name Attributes.phoneFootprintStrengthOverall": {"Low": 0, "Medium": 1, "High": 2, "Very High": 3},
    "Email Name Attributes.emailFootprintStrength": {"Very Low": 0, "Low": 1, "High": 2, "Very High": 3}
}

def encode_ordinals(data, ordinal_maps=ORDINAL_MAPS):
    """Convert ordinal string values to integers, using the provided mapping."""
    out = data.copy()
    out["footprint_no_records"] = out["Email Attributes.footprint"].eq("NO_RECORDS").astype("int8")  # special case for the first column
    for col, mapping in ordinal_maps.items():
        out[col] = out[col].map(mapping)
    return out

dev_o = encode_ordinals(dev)
print(dev_o[list(ORDINAL_MAPS)].isna().sum())          # expect 17, 0, 0  (only NO_RECORDS -> NaN)
print(dev_o["footprint_no_records"].sum())              # expect 17

In [ ]:
CATEGORY_LEVELS = {
    # column: (levels to keep, keep missing as its own level?)
    "Phone Network.numberBillingType":           (["prepaid", "postpaid", "unknown"], True),
    "Phone Network.currentNetworkName":          (["Airtel", "Reliance Jio"],         False),
    "Email Attributes.emailFinalRecommendation": (["Good", "Check"],                  False),
}

def clean_categories(data, levels=CATEGORY_LEVELS):
    """Keep listed levels; everything else becomes 'other' (missing becomes 'missing' if requested)."""
    out = data.copy()
    for col, (keep, missing_separate) in levels.items():
        s = out[col]
        cleaned = s.where(s.isin(keep), "other")              # not in the list -> 'other'
        if missing_separate:
            cleaned = cleaned.where(s.notna(), "missing")     # NaN -> 'missing'
        out[col] = cleaned
    return out

dev_c = clean_categories(dev_o)
for col in CATEGORY_LEVELS:
    print(dev_c[col].value_counts().to_dict())

The kept levels are fixed lists chosen from dev, so dev, holdout and test are encoded identically. A rule recomputed on each dataset could label the same category differently in test.

In [ ]:
NUMERIC_FEATURES = [
    "Email Attributes.digitalAge", "Email Name Attributes.digitalage",
    "Phone Name Attributes.phoneNameDigitalAge", "Email Name Attributes.nameMatchScore",
    "Email Name Attributes.nameEmailMatch", "Email Attributes.uniqueHits",
    "email_platform_share", "phone_platform_share",
    "Email Attributes.footprint", "Phone Name Attributes.phoneFootprintStrengthOverall",
    "Email Name Attributes.emailFootprintStrength",                     # ordinal codes
]
FLAG_FEATURES = [
    "Phone Social Advance.whatsapp", "Phone Network.numberHasPortingHistory",
    "Email Name Attributes.firstNameMatch", "Email Name Attributes.lastNameMatch",
    "email_name_not_found", "email_not_found", "phone_name_not_found", "footprint_no_records",
]
CATEGORICAL_FEATURES = [
    "Phone Network.numberBillingType", "Phone Network.currentNetworkName",
    "Email Attributes.emailFinalRecommendation",
]
MODEL_FEATURES = NUMERIC_FEATURES + FLAG_FEATURES + CATEGORICAL_FEATURES

def build_features(data, sentinel_mode="flag"):
    """Apply all feature steps in order and return UID + model features (+ Target if present)."""
    out = handle_sentinels(data, mode=sentinel_mode)
    out = add_platform_share(out)
    out = encode_ordinals(out)
    out = clean_categories(out)

    features = MODEL_FEATURES if sentinel_mode == "flag" else \
               [f for f in MODEL_FEATURES if f not in NOT_FOUND_FLAGS]   # raw mode has no not-found flags
    keep = ["UID"] + features + (["Target"] if "Target" in out.columns else [])
    return out[keep]

dev_fe = build_features(dev)
print(dev_fe.shape)
dev_fe.head(3).T

In [ ]:
holdout_fe = build_features(holdout)
test_fe    = build_features(test_raw)

# 1. Same columns everywhere (test has no Target)
assert list(holdout_fe.columns) == list(dev_fe.columns)
assert list(test_fe.columns) == [c for c in dev_fe.columns if c != "Target"]

# 2. Every row kept, same order, same UIDs (brief requirement for test)
for raw, fe in [(dev, dev_fe), (holdout, holdout_fe), (test_raw, test_fe)]:
    assert len(fe) == len(raw)
    assert (fe["UID"].values == raw["UID"].values).all()

# 3. No -1 left in any numeric model feature (flag mode)
assert not dev_fe[NUMERIC_FEATURES].eq(-1).any().any()

# 4. Categories: only the expected levels appear (no surprise text in test)
for col, (keep, missing_separate) in CATEGORY_LEVELS.items():
    allowed = set(keep) | {"other"} | ({"missing"} if missing_separate else set())
    for name, fe in [("dev", dev_fe), ("holdout", holdout_fe), ("test", test_fe)]:
        assert set(fe[col].unique()) <= allowed, f"{name}: unexpected level in {col}"

print("All checks passed.")

# 5. Missing values per feature: dev vs test (where will the imputer have to work?)
na_compare = pd.DataFrame({
    "dev_missing%":  dev_fe[MODEL_FEATURES].isna().mean().mul(100).round(1),
    "test_missing%": test_fe[MODEL_FEATURES].isna().mean().mul(100).round(1),
})
na_compare[na_compare.max(axis=1) > 0]